# 하이퍼파라미터 튜닝

Supabase의 `ready` 데이터셋 shard를 검증해 읽고, validation split으로만 모델과 하이퍼파라미터를 선택한다. test split은 최종 설정에 한 번만 사용한다. 이 노트북은 run·artifact를 생성하지 않는 읽기 전용 연구 도구다.

## 실행 전 조건

- 저장소 `.env`에 `SUPABASE_URL`과 서버 전용 Supabase 키가 있어야 한다.
- `uv sync --extra train`으로 PyTorch와 scikit-learn 선택 의존성을 설치한다.
- 데이터셋 최신 진단이 `failed`이면 loader가 실행을 차단한다.
- 빠른 탐색에서는 샘플 상한과 epoch를 작게 두고, 후보가 좁혀진 뒤 전체 데이터로 재검증한다.

In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from pivot.config import TrainingConfig
from pivot.models import build_model
from pivot.storage.datasets import DatasetRepository
from pivot.storage.diagnostics import DiagnosticReportRepository
from pivot.storage.supabase import PostgrestClient, StorageObjectClient
from pivot.training.evaluate import evaluate_model
from pivot.training.runs import build_split_datasets
from pivot.training.train import make_loader, select_device, train_model

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
CACHE_ROOT = ROOT / 'data' / 'tmp' / 'notebook-shards'


In [ ]:
db = PostgrestClient()
datasets = DatasetRepository(db)
diagnostics = DiagnosticReportRepository(db)
storage = StorageObjectClient()

ready_datasets = [row for row in datasets.list() if row['status'] == 'ready']
catalog = pd.DataFrame(ready_datasets)
columns = [
    column
    for column in ['id', 'name', 'timeframe', 'sample_count', 'class_counts', 'created_at']
    if column in catalog.columns
]
display(catalog[columns] if not catalog.empty else catalog)

DATASET_ID = int(ready_datasets[0]['id']) if ready_datasets else None
if DATASET_ID is None:
    raise RuntimeError('ready 데이터셋이 없습니다.')
print('선택 데이터셋:', DATASET_ID)

In [ ]:
splits = build_split_datasets(
    datasets, storage, diagnostics, DATASET_ID, CACHE_ROOT
)
{
    split: {
        'samples': len(dataset),
        'features': dataset.feature_columns,
        'class_counts': pd.Series(dataset.labels()).value_counts().sort_index().to_dict(),
    }
    for split, dataset in splits.items()
}

In [ ]:
class DatasetView:
    def __init__(self, dataset, limit: int | None):
        self.dataset = dataset
        self.feature_columns = dataset.feature_columns
        if limit is None or len(dataset) <= limit:
            self.indices = list(range(len(dataset)))
        else:
            self.indices = np.linspace(0, len(dataset) - 1, limit, dtype=int).tolist()
        all_labels = dataset.labels()
        self._labels = [all_labels[index] for index in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return self.dataset[self.indices[index]]

    def labels(self):
        return list(self._labels)

MAX_TRAIN_SAMPLES = 5000
MAX_VALIDATION_SAMPLES = 2000
train_data = DatasetView(splits['train'], MAX_TRAIN_SAMPLES)
validation_data = DatasetView(splits['validation'], MAX_VALIDATION_SAMPLES)
test_data = splits['test']
len(train_data), len(validation_data), len(test_data)

In [ ]:
EPOCHS = 3
SEED = 42
MAX_TRIALS = 12

search_space = [
    {
        'model': model,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'sampler': sampler,
        'epochs': EPOCHS,
        'seed': SEED,
    }
    for learning_rate, batch_size, sampler, model in product(
        [3e-4, 1e-3, 3e-3],
        [64, 256],
        ['none', 'weighted'],
        ['cnn1d_legacy_v1', 'cnn1d_temporal_v1'],
    )
][:MAX_TRIALS]
pd.DataFrame(search_space)

In [ ]:
device = select_device()
trial_rows = []
best_model = None
best_config = None
best_validation_f1 = -1.0

for trial, values in enumerate(search_space):
    config = TrainingConfig.model_validate(values)
    model = build_model(
        config.model, input_size=len(train_data.feature_columns), output_size=3
    )
    trained = train_model(
        model, train_data, validation_data, config, device=device
    )
    validation = evaluate_model(
        trained['model'], make_loader(validation_data, config, training=False), device
    )['metrics']
    row = {
        'trial': trial,
        **values,
        'best_epoch': trained['best_epoch'],
        'validation_loss': validation['loss'],
        'validation_accuracy': validation['accuracy'],
        'validation_macro_f1': validation['macro_f1'],
        'low_precision': validation['per_class_metrics']['0']['precision'],
        'high_precision': validation['per_class_metrics']['1']['precision'],
        'ignore_f1': validation['per_class_metrics']['2']['f1'],
    }
    trial_rows.append(row)
    if validation['macro_f1'] > best_validation_f1:
        best_validation_f1 = validation['macro_f1']
        best_model = trained['model']
        best_config = config
    if device.type == 'mps':
        torch.mps.empty_cache()

trials = pd.DataFrame(trial_rows).sort_values(
    ['validation_macro_f1', 'validation_loss'], ascending=[False, True]
)
display(trials)

In [ ]:
# validation으로 선택한 단 하나의 모델만 test에서 평가한다.
if best_model is None or best_config is None:
    raise RuntimeError('완료된 trial이 없습니다.')

test_result = evaluate_model(
    best_model, make_loader(test_data, best_config, training=False), device
)['metrics']
display(pd.Series({
    'model': best_config.model,
    'learning_rate': best_config.learning_rate,
    'batch_size': best_config.batch_size,
    'sampler': best_config.sampler,
    'validation_macro_f1': best_validation_f1,
    'test_loss': test_result['loss'],
    'test_accuracy': test_result['accuracy'],
    'test_macro_f1': test_result['macro_f1'],
}))
display(pd.DataFrame(test_result['per_class_metrics']).T)
display(pd.DataFrame(
    test_result['confusion_matrix'],
    index=['actual_0', 'actual_1', 'actual_2'],
    columns=['pred_0', 'pred_1', 'pred_2'],
))

## 승격 기준

- 축소 샘플 탐색의 상위 후보만 전체 train/validation 데이터로 다시 학습한다.
- 최종 후보는 최소 3개 seed에서 validation macro F1의 평균과 표준편차를 확인한다.
- test는 후보 선택에 사용하지 않는다. 최종 설정 확정 후 한 번만 보고한다.
- 고점·저점 precision이 baseline보다 악화되면 전체 accuracy가 높아도 채택하지 않는다.
- 채택 설정은 UI에서 정식 run으로 실행해 Supabase snapshot과 검증된 checkpoint를 남긴다.